# 26 RRT：随机树探索连续空间


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**RRT**（Rapidly-exploring Random Tree，LaValle 1998）在 **连续** 状态空间里长一棵树，用随机采样把树拉进未探索区域。本课状态就是平面点 \((x,y)\in[0,1]^2\)，障碍是圆。没有格子、没有现成邻居列表。

- **输入**：场景 `{width,height,obstacles,start,goal}`；迭代上限 n；步长 \(\varepsilon\)；目标偏置 `goal_bias`；随机种子。
- **输出：** `found`，折线 `path`，代价（折线欧氏长），树的 `nodes` 与 `parent`。

一轮迭代：

1. **Sample：** 以概率 `goal_bias` 取 goal，否则在矩形里均匀随机。
2. **Nearest：** 树上离样本最近的节点 \(x_{\mathrm{near}}\)（教学用暴力扫）。
3. **Steer：** 从 \(x_{\mathrm{near}}\) 向样本走至多 \(\varepsilon\) 得到 \(x_{\mathrm{new}}\)。
4. **Collision：** 检查 **线段** \(x_{\mathrm{near}}x_{\mathrm{new}}\) 是否穿障，不是只查两个端点。
5. 无碰则把 \(x_{\mathrm{new}}\) 入树。若它离 goal 不到 \(\varepsilon\) 且连 goal 的线段也自由，把 goal 挂上，成功。

---

## 2. 和谁相邻

|  | RRT | Dijkstra / A\* | PRM | RRT\*（27） |
|--|-----|-----------------|-----|--------------|
| 空间 | 连续 | 已离散的图 | 连续，先建路网再搜 | 连续 |
| 邻居 | steer 造出来 | 图的出边 | 半径内连边 | steer + Near |
| 最优 | 否 | 是（非负权） | 取决于路网 | 渐近最优 |
| 完备 | 概率完备 | 图连通则完备 | 概率完备 | 概率完备 |

**已经是路网 / 格子就不要用 RRT**，那是 Dijkstra 的活。RRT 用在没有 \(V,E\) 的连续障碍空间。Voronoi 偏向让树伸进开阔地，而不是像 BFS 一圈圈胀。

---

## 3. 不变量 / 定理

- 树上每条边都是曾经通过线段碰撞检查的 steer 段。因此整条根到叶的折线（在「线段检测不漏」的前提下）不碰圆。
- **概率完备：** 障碍之间的缝有正宽度、\(\varepsilon\) 足够小、n→∞ 时成功概率 → 1。不是有限 n 的保证。
- **不必最优。** 先碰到 goal 的那条折线往往锯齿、绕远。换 seed 路径形状会变。
- 检测不变量：若只查端点，存在整段穿过圆、两端在圆外的假阴性（第 6 节）。

线段检测：段上 21 点（`i=0..20`，\(t=i/20\)），任一点进圆则撞。圆相对步长不太小时够用。

---

## 4. 完整伪代码（与 `mini_rrt` 对应）

```
function PointInCircle(p, ox, oy, r):
    return hypot(p-center) <= r

function SegCollides(a, b, steps=20):
    for i in 0..steps:
        p ← (1-t)*a + t*b,  t=i/steps
        if PointInCircle(p): return True
    return False

function Steer(a, b, ε):
    d ← hypot(b-a)
    if d <= ε: return b
    return a + ε * (b-a)/d

function MiniRRT(start, goal, n, ε, seed):
    rng ← Random(seed)
    nodes ← [start];  parent ← [None]
    for k in 1..n:
        sample ← goal if rng.random() < goal_bias else Uniform([0,1]^2)
        i ← argmin_j ||nodes[j]-sample||
        x_new ← Steer(nodes[i], sample, ε)
        if x_new 出界 or SegCollides(nodes[i], x_new): continue
        parent.append(i); nodes.append(x_new)
        if hypot(x_new, goal) < ε and not SegCollides(x_new, goal):
            parent.append(len(nodes)-1); nodes.append(goal)
            return SUCCESS, Reconstruct, Length
    return FAILURE
```

---

## 5. 复杂度

暴力最近邻合计 \(O(n^2)\)，每次碰撞 \(O(\mathrm{steps}\cdot|\mathrm{obs}|)\)。KD 树可到约 \(O(n\log n)\)。\(\varepsilon\) 太大穿不过细缝，太小则 n=400 爬不到 goal。

---

## 6. 完整手算 / 小表

单位方、一圆心 (0.5,0.5) 半径 0.2。起点 (0.1,0.1)，终点 (0.9,0.9)。直线穿过圆，所以成功路径必须绕上或绕下。

**只查端点会漏的线段：** a=(0.2,0.5)，b=(0.8,0.5)。

| 检查 | 结果 |
|------|------|
| a 在圆内？ | hypot 到圆心=0.3 > 0.2，否 |
| b 在圆内？ | 同样 0.3 > 0.2，否 |
| 中点 (0.5,0.5) | 圆心，**在圆内** |
| 20 点采样 | 必能抓到中点附近 |

所以碰撞函数必须采样线段。下面代码先打印这张表，再跑 n=400、ε=0.08、seed=0 的整棵树。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

从碰撞、steer、最近邻写到主循环。打印是否找到、代价、节点数；前几个新节点也打印。


In [ ]:
import math
import random

START, GOAL = (0.1, 0.1), (0.9, 0.9)
OX, OY, ORAD = 0.5, 0.5, 0.2
STEP, N, GOAL_BIAS = 0.08, 400, 0.08

def point_in_circle(p):
    return math.hypot(p[0] - OX, p[1] - OY) <= ORAD

def seg_collides(a, b, steps=20):
    # 含端点 i=0 与 i=steps
    for i in range(steps + 1):
        t = i / steps
        p = (a[0] * (1 - t) + b[0] * t, a[1] * (1 - t) + b[1] * t)
        if point_in_circle(p):
            return True
    return False

def steer(a, b, step):
    dx, dy = b[0] - a[0], b[1] - a[1]
    dist = math.hypot(dx, dy)
    if dist <= step:
        return b
    return (a[0] + step * dx / dist, a[1] + step * dy / dist)

def nearest_idx(nodes, p):
    best, bi = 1e9, 0
    for i, q in enumerate(nodes):
        d = (q[0] - p[0]) ** 2 + (q[1] - p[1]) ** 2
        if d < best:
            best, bi = d, i
    return bi

a, b = (0.2, 0.5), (0.8, 0.5)
print("端点碰撞", point_in_circle(a), point_in_circle(b), "线段采样", seg_collides(a, b))

def reconstruct(parent, nodes, idx):
    chain = []
    while idx is not None:
        chain.append(nodes[idx])
        idx = parent[idx]
    chain.reverse()
    return chain

def path_cost(path):
    return sum(math.hypot(q[0] - p[0], q[1] - p[1]) for p, q in zip(path, path[1:]))

def mini_rrt(n=N, step=STEP, seed=0, goal_bias=GOAL_BIAS, verbose=True):
    rng = random.Random(seed)
    nodes, parent = [START], [None]
    for k in range(n):
        sample = GOAL if rng.random() < goal_bias else (rng.random(), rng.random())
        i = nearest_idx(nodes, sample)
        nxt = steer(nodes[i], sample, step)
        if not (0.0 <= nxt[0] <= 1.0 and 0.0 <= nxt[1] <= 1.0):
            continue
        if seg_collides(nodes[i], nxt):
            continue
        parent.append(i)
        nodes.append(nxt)
        if verbose and len(nodes) <= 6:
            print(f"  add #{len(nodes)-1} {nxt}  parent={i}")
        if math.hypot(nxt[0] - GOAL[0], nxt[1] - GOAL[1]) < step and not seg_collides(nxt, GOAL):
            parent.append(len(nodes) - 1)
            nodes.append(GOAL)
            path = reconstruct(parent, nodes, len(nodes) - 1)
            return {"found": True, "path": path, "cost": path_cost(path), "nodes": nodes, "parent": parent}
    return {"found": False, "path": [], "cost": float("inf"), "nodes": nodes, "parent": parent}

r = mini_rrt()
print("found", r["found"], "cost", r["cost"], "nodes", len(r["nodes"]))


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `Random(seed)` | 可复现 |
| `goal_bias` | 否则很难采到终点邻域 |
| `nearest` 平方距离 | 比大小不必开方 |
| `steer` 截到 ε | 新边有上限 |
| `seg_collides` 21 点 | 只查端点会穿过圆 |
| 近 goal 再查最后一跳 | 那一段也要无碰 |

seed=0、n=400、ε=0.08 应找到；代价约 1.52（直线 \(\sqrt{2}\approx 1.41\)）。


In [ ]:
from lib.maps import continuous_scene
from lib.algos.rrt import rrt

coarse = mini_rrt(n=400, step=0.5, seed=0, verbose=False)
fine = mini_rrt(n=400, step=0.01, seed=0, verbose=False)
print("ε=0.5 过粗 found", coarse["found"], "nodes", len(coarse["nodes"]))
print("ε=0.01 过细 found", fine["found"], "nodes", len(fine["nodes"]))

scene = continuous_scene()
lib = rrt(scene, seed=1, n=1500)
print("库 rrt seed=1 n=1500 found", lib["found"], "cost", lib["cost"], "nodes", len(lib["nodes"]))


## 9. 常见 bug

1. **只检查端点。** 水平穿过圆心的线段两端都自由，树会穿障。
2. **ε 太大。** 一步跳过窄缝，或新段太长很难绕圆；本课 0.5 往往树很稀。
3. **ε 太小。** n 不够，爬不到 (0.9,0.9)。
4. **没有 goal bias。** 均匀采样打中终点邻域的概率是面积阶，收敛慢。
5. **已离散路网还跑 RRT。** 浪费，且不最优；用 Dijkstra。

## 10. 对照库

`lib/algos/rrt.py`：`_collides` 多圆、`_seg_collides` 20 步、`_steer`、`_nearest` 暴力、`rrt(..., seed, n, step, goal_bias)`。`continuous_scene()` 是单位方、三圆、start [0.08,0.08]、goal [0.92,0.92]。eval `test_26` 固定 seed=1、n=1500 必须找到。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.show_rrt(scene, lib, "RRT")
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 空间 | 连续，邻居靠 steer 生成 |
| 保证 | 概率完备，**非**最优 |
| 碰撞 | 线段采样，不只端点 |
| 复杂度 | 暴力最近邻 \(O(n^2)\) |
| 下一课 | 选父 + rewire → RRT\* |

## 练习

1. 证明 a=(0.2,0.5)、b=(0.8,0.5) 两端在 r=0.2 的圆外、中点在圆内。只查端点的 RRT 会怎样？
2. 把 ε 改成 0.5 和 0.01（n=400, seed=0），记录 found 与节点数。
3. 为何 `topo_oneway` 这种有向路网不该改写成 RRT？
